# CN-GongWen Benchmark · Colab 一键复现 Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/SH-PA/blob/main/notebooks/CN_GongWen_Colab_Pipeline.ipynb)

端到端复现两套**中国党政机关公文**评测数据集（对标《党政机关公文处理工作条例》与 GB/T 9704—2012）：
克隆仓库 → 安装依赖 →（可选）配置 MiniMax → 生成 → 严格校验 → 单元测试 → 分布可视化 → 证据复核 → 打分器演示。

> 所有机关、字号、内容均为合成示例，不对应真实单位或真实公文。数值/合规标签均由 Python 确定性计算。


## 1️⃣ 克隆仓库


In [ ]:
import os
if not os.path.exists('SH-PA'):
    !git clone https://github.com/pariskang/SH-PA.git
%cd SH-PA
!git log --oneline -3


## 2️⃣ 安装依赖（含中文字体）

核心生成与校验无需第三方库；此处额外安装 pytest、PyYAML 以便测试与读取规则配置，并安装 Noto CJK 字体、配置 matplotlib 以正确显示图表中文。

In [ ]:
!pip -q install pytest PyYAML
!apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1 || true
# 可选：LLM 与 Parquet 支持
# !pip -q install 'litellm>=1.0.0' pandas pyarrow

# 配置中文字体（修复图表中文乱码）
import glob, matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
cands = []
for pat in ('NotoSansCJK*', 'NotoSerifCJK*', 'wqy*', '*CJK*'):
    cands += glob.glob(f'/usr/share/fonts/**/{pat}', recursive=True)
cjk = None
for p in cands:
    try:
        fm.fontManager.addfont(p); cjk = fm.FontProperties(fname=p).get_name(); break
    except Exception:
        continue
if cjk:
    plt.rcParams['font.sans-serif'] = [cjk, 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('中文字体：', cjk or '未找到（图表中文可能显示为方块，可重跑本步）')

## 3️⃣ （可选）配置 MiniMax API

LiteLLM 仅用于**事实护栏下**的问题改写/播报润色，不改变文种、字号、日期、密级等关键事实；
因此有无 LLM 产物完全一致。不配置可直接跳过本节。


In [ ]:
import os
# os.environ['MINIMAX_API_KEY']  = 'sk-...'
# os.environ['MINIMAX_API_BASE'] = 'https://your-openai-compatible-relay/v1'
# os.environ['MINIMAX_MODEL']    = 'MiniMax-M1'
USE_LLM = bool(os.getenv('MINIMAX_API_KEY'))
print('使用 LiteLLM 改写：', USE_LLM)


## 4️⃣ 生成两套数据集（standard 档）


In [ ]:
cmd = 'python gongwen_benchmark/scripts/generate_benchmarks.py --profile standard'
if USE_LLM: cmd += ' --use-litellm'
print(cmd)
!{cmd}


## 5️⃣ 严格跨文件校验


In [ ]:
!python gongwen_benchmark/scripts/validate_artifacts.py


## 6️⃣ 单元测试


In [ ]:
!pytest -q


## 7️⃣ 分布可视化

问题类型 / 任务类型 / Q 难度分布。


In [ ]:
import json, collections
import matplotlib.pyplot as plt
ROOT='gongwen_benchmark'
def load(p): return [json.loads(l) for l in open(p,encoding='utf-8') if l.strip()]
hidden=load(f'{ROOT}/dataset_1_question_only/questions_with_hidden_metadata.jsonl')
ques=load(f'{ROOT}/dataset_2_data_qa/questions.jsonl')
qt=collections.Counter(x['question_type'] for x in hidden)
tt=collections.Counter(x['task_type'] for x in ques)
df=collections.Counter(x['difficulty'] for x in hidden)
fig,ax=plt.subplots(1,3,figsize=(18,5))
ax[0].barh(list(qt.keys()),list(qt.values())); ax[0].set_title('CN-GongWen-Q 问题类型')
ax[1].barh(list(tt.keys()),list(tt.values())); ax[1].set_title('CN-GongWen-DataQA 任务类型')
ax[2].bar(list(df.keys()),list(df.values())); ax[2].set_title('Q 难度分布')
plt.tight_layout(); plt.show()
print('hard 占比：', round(df['hard']/sum(df.values()),3))


## 7️⃣·医疗政策方向分布

约一半内容为医疗卫生政策方向，覆盖 16 个医疗政策子领域。


In [ ]:
import csv, collections
recs=list(csv.DictReader(open(f'{ROOT}/dataset_2_data_qa/records.csv',encoding='utf-8')))
dom=collections.Counter(r['policy_domain'] for r in recs)
area=collections.Counter(r['medical_area'] for r in recs if r['medical_area'])
hid=load(f'{ROOT}/dataset_1_question_only/questions_with_hidden_metadata.jsonl')
qdom=collections.Counter(x['policy_domain'] for x in hid)
fig,ax=plt.subplots(1,2,figsize=(16,6))
ax[0].bar(['语料·'+k for k in dom]+['Q·'+k for k in qdom], list(dom.values())+list(qdom.values()))
ax[0].set_title('政策领域分布（语料 vs Q，各约50%医疗）'); ax[0].tick_params(axis='x',rotation=30)
ax[1].barh(list(area.keys()),list(area.values())); ax[1].set_title('16个医疗政策子领域·语料发文量')
plt.tight_layout(); plt.show()
print('语料医疗占比：', round(dom['医疗卫生']/sum(dom.values()),3), '| Q医疗占比：', round(qdom['医疗卫生']/sum(qdom.values()),3), '| 子领域覆盖：', len(area))


## 8️⃣ 证据行复核

确认每道 DataQA 题的证据行均存在于 records.csv。


In [ ]:
import csv
doc_ids={r['doc_id'] for r in csv.DictReader(open(f'{ROOT}/dataset_2_data_qa/records.csv',encoding='utf-8'))}
ans=load(f'{ROOT}/dataset_2_data_qa/answers.jsonl')
bad=[a['question_id'] for a in ans if not set(a['evidence_rows'])<=doc_ids]
print('记录数：',len(doc_ids),'| 答案数：',len(ans),'| 证据越界题：',len(bad))
assert not bad, bad[:5]


## 9️⃣ 打分器演示

以金标准自评作连通性检查，理想分数应为 1.0。


In [ ]:
from gongwen_benchmark.evaluation.scorer import dataset1_score, dataset2_score
from pathlib import Path
R=Path(ROOT)
s1=dataset1_score(R/'dataset_1_question_only/questions_with_hidden_metadata.jsonl', R/'dataset_1_question_only/questions_with_hidden_metadata.jsonl')
s2=dataset2_score(R/'dataset_2_data_qa/answers.jsonl', R/'dataset_2_data_qa/answers.jsonl')
print('Q 自评：', json.dumps(s1,ensure_ascii=False,indent=2))
print('DataQA 自评：', json.dumps(s2,ensure_ascii=False,indent=2))


## 🔟 写作测试 prompt（dataset_3，按 token 分桶）

短/中/长按**目标产出 token** 分桶，覆盖 15 法定文种，蕴含复杂行文框架与行文规则（标题三要素、层次序数、上行文签发人、请示单一主送等）、内容可执行性与语言安全；打分器 11 维结构化合规，金标准自评满分。

In [ ]:
# CN-GongWen-Writing（dataset_3）：按目标产出 token 分桶的公文写作测试 prompt
import collections
from pathlib import Path
from gongwen_benchmark.evaluation.scorer import dataset3_writing_score
from gongwen_benchmark.scripts.tokens import estimate_tokens
from gongwen_benchmark.scripts.generate_writing_prompts import LENGTH_BUCKETS as LB
W=f'{ROOT}/dataset_3_writing'
wrub=load(f'{W}/writing_prompts_with_rubric.jsonl')
order=['short','medium','long']
bucket=collections.Counter(h['length_bucket'] for h in wrub)
dtc=collections.Counter(h['spec']['doc_type'] for h in wrub)
toks={b:[estimate_tokens(h['reference_answer']) for h in wrub if h['length_bucket']==b] for b in order}
fig,ax=plt.subplots(1,3,figsize=(20,5))
ax[0].bar([f"{b}\n{LB[b]['target_tokens']}" for b in order],[bucket[b] for b in order]); ax[0].set_title('长度分桶题量（短/中/长·目标token）')
ax[1].barh(list(dtc.keys()),list(dtc.values())); ax[1].set_title('15 法定文种覆盖')
ax[2].boxplot([toks[b] for b in order]); ax[2].set_xticks([1,2,3]); ax[2].set_xticklabels(order); ax[2].set_title('参考公文实际 token 分布')
plt.tight_layout(); plt.show()
P=Path(W)/'writing_prompts_with_rubric.jsonl'; sc=dataset3_writing_score(P,P)
print('题量：',len(wrub),'| 桶：',dict(bucket),'| 文种：',len(dtc))
print('写作打分自评（11 维，应全 1.0）：', json.dumps(sc, ensure_ascii=False))

## ✅ 结语

至此已生成并校验两套公文评测数据集。将你的模型预测写成与金标准同 `question_id` 的 JSONL，
再用 `gongwen_benchmark/evaluation/scorer.py` 即可离线评分。
